In [ ]:
# Launcher: clone the repo if missing, otherwise pull, then compile, run, display.
import glob, os, shutil, subprocess, sys
from IPython.display import display
from PIL import Image

REPO_URL = "https://github.com/ShreyasxAnand/pointwalk"
BRANCH = "dev"
FLAGS = ["-O2", "-arch=sm_75", "-fmad=false"]  # -fmad=false: CPU and GPU match bit for bit
PROGRAMS = ["project", "bench", "pano"]  # src/<name>.cu -> build/<name>; pano reads project's scene.bin

# Waymo (step 7 on). A few frames of one segment are converted here on Colab,
# straight from Waymo's bucket, and kept outside the repo. Nothing is committed.
WAYMO_INPUT = "gs://waymo_open_dataset_v_2_0_1/training"
WAYMO_SEGMENT = "10023947602400723454_1120_000_1140_000"
WAYMO_FRAMES = 2                       # how many frames to convert; 0 skips everything Waymo
WAYMO_PROGRAMS = ["range_to_points"]   # each is run as ./<name> <converted folder>
WAYMO_OUT = "/content/waymo_out/" + WAYMO_SEGMENT   # the converted files
WAYMO_PKGS = "/content/waymo_pkgs"     # the converter's pinned packages, kept apart from Colab's own

assert "<" not in REPO_URL, "Set REPO_URL at the top of this cell first."
repo = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
build = os.path.join(repo, "build")


def sh(*cmd, cwd=None, env=None):
    """Run a command, print its output, stop the cell if it fails."""
    print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=cwd, env=env, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout, flush=True)
    if r.returncode != 0:
        raise SystemExit(f"FAILED (exit {r.returncode}): {' '.join(cmd)}")


# 1. Get the code.
if not os.path.isdir(os.path.join(repo, ".git")):
    sh("git", "clone", "--branch", BRANCH, REPO_URL, repo)
else:
    sh("git", "fetch", "origin", BRANCH, cwd=repo)
    sh("git", "checkout", BRANCH, cwd=repo)
    sh("git", "pull", "--ff-only", "origin", BRANCH, cwd=repo)
sh("git", "log", "-1", "--format=%h %ad %s", "--date=iso", cwd=repo)  # what is being run
sh("nvidia-smi", "--query-gpu=name", "--format=csv,noheader")

# 2. Compile and run each program from a fresh build/ (outputs land there, untracked).
shutil.rmtree(build, ignore_errors=True)
os.makedirs(build)
for name in PROGRAMS:
    sh("nvcc", *FLAGS, os.path.join(repo, "src", name + ".cu"), "-o", os.path.join(build, name))
    sh("./" + name, cwd=build)

# 3. Waymo: convert a few frames (only if they are not there yet), then run the Waymo programs on them.
if WAYMO_FRAMES > 0:
    last_file = os.path.join(WAYMO_OUT, "range", f"frame{WAYMO_FRAMES - 1:04d}_laser1.bin")
    if not os.path.exists(last_file):
        from google.colab import auth
        auth.authenticate_user()  # sign in with the Google account that has access to the Waymo dataset
        sh(sys.executable, "-m", "pip", "install", "-q", "--target", WAYMO_PKGS,
           "-r", os.path.join(repo, "waymo", "requirements.txt"))
        sh(sys.executable, os.path.join(repo, "waymo", "convert.py"), "--input", WAYMO_INPUT,
           "--segment", WAYMO_SEGMENT, "--out", WAYMO_OUT, "--frames", str(WAYMO_FRAMES),
           env={**os.environ, "PYTHONPATH": WAYMO_PKGS})
    for name in WAYMO_PROGRAMS:
        sh("nvcc", *FLAGS, os.path.join(repo, "src", name + ".cu"), "-o", os.path.join(build, name))
        sh("./" + name, WAYMO_OUT, cwd=build)

# 4. Display any PGM/PPM images the programs wrote.
for path in sorted(glob.glob(os.path.join(build, "*.p[gp]m"))):
    print(os.path.basename(path))
    display(Image.open(path))